# Upload cropped volume and meshes to analyze wall heterogenity
Loads a `.vti`/`.vtr` reconstruction, crops the z-stack to the true material thickness (symmetric around a known center voxel), and plots the center-y slice (Mz contrast) before/after to verify the crop landed on the right region.

In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti



from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.measure import marching_cubes
import pyvista as pv

import matplotlib.ticker as mticker
import pandas as pd

import json

import Functions_3D as vis
import curvature_segmentation as curve

from scipy import stats

from scipy.optimize import curve_fit
from matplotlib.gridspec import GridSpec


from scipy.spatial import cKDTree
from scipy.ndimage import map_coordinates

from skimage import measure
from scipy.ndimage import gaussian_filter1d



# Load the volume

In [ ]:
def load_volume_analysis(save_dir, sample_name, date, smooth):
    """
    Load back everything saved by save_volume_analysis: cropped volume,
    smoothed volume (if present), and all derived-field dicts, fully
    reconstructed including their non-array metadata (shape, spacing, etc).

    Returns
    -------
    cropped : dict
    volume_smoothed : dict or None
    fields : dict of dict
        e.g. fields['jacobian'], fields['curl'], fields['divergence'], etc.,
        each reconstructed as a full dict just like the original.
    metadata : dict
        Top-level metadata, useful for provenance/sanity checks.
    """
    meta_path = os.path.join(save_dir, f'{sample_name}_volume_analysis_metadata_{date}.json')
    with open(meta_path) as f:
        metadata = json.load(f)

    if smooth: 
        # --- Smoothed volume, if it was saved ---
        smoothed_path = os.path.join(save_dir, f'{sample_name}_smoothed_{date}.npz')
        volume = None
        if os.path.exists(smoothed_path):
            smoothed_npz = np.load(smoothed_path)
            volume = {
                'Mx': smoothed_npz['Mx'], 'My': smoothed_npz['My'], 'Mz': smoothed_npz['Mz'],
                'magnitude': smoothed_npz['magnitude'],
                'shape': tuple(smoothed_npz['shape']),
                'sigma': metadata.get('sigma'),
            }

    else:

        cropped_npz = np.load(os.path.join(save_dir, f'{sample_name}_volume_{date}.npz'))
        volume = {
            'Mx': cropped_npz['Mx'], 'My': cropped_npz['My'], 'Mz': cropped_npz['Mz'],
            'magnitude': cropped_npz['magnitude'],
            'shape': tuple(cropped_npz['shape']),
        }


    # --- Each derived-field dict ---
    fields = {}
    for field_name in metadata.get('saved_fields', []):
        field_path = os.path.join(save_dir, f'{sample_name}_{field_name}_{date}.npz')
        field_npz = np.load(field_path)
        field_dict = {key: field_npz[key] for key in field_npz.files}
        field_meta = metadata.get(f'{field_name}_meta', {})
        for k, v in field_meta.items():
            field_dict[k] = tuple(v) if isinstance(v, list) else v
        fields[field_name] = field_dict

    return volume, fields, metadata

In [ ]:
sample_names = ['452', '270', '105', '665'] 
#dates = ['mag_const_updated_interpolation', '14AUG26_interpolated', '22AUG26_interpolated', 'mag_const_TV_interpolated']
#dates = ['mag_const_NOTinterpolated_48', '14AUG26_interpolated', '22AUG26_interpolated', '22AUG26_interpolated']

#dates = ['exactsubset_mag_const_TV_interpolated_OCT26', '14AUG26_interpolated', '22AUG26_interpolated', 'mag_const_TV_interpolated']
#dates = ['normalized_interpolated','normalized_interpolated'  ]

# dates = ['mag_const_TV_sqrt(2)_interp_OCT26', '14AUG26_interpolated', 'mag_const_TV_interpolated_t_OCT26', '22AUG26_interpolated']

#dates = ['cropped_mag_const_2interp_bsd_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26']


################ Cropped from full volume to center, with TV, no interpolation ################
# dates = ['TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26'] 
dates = ['TV_5mask_cropped', 'TV_9thick_cropped', 'TV_5thick_cropped', 'TV_8nm_NOV26'] 
dz, dy, dx = 8.0, 8.0, 8.0  
voxel_nm = dz


sample_colors = ['limegreen', 'deeppink', 'cyan', 'gold']
use_smooth = [False, False, False, False]
thickness_list_nm = [33, 66, 45,51]


sample = 0
thickness = thickness_list_nm[sample]
sample_name = sample_names[sample]
date = dates[sample]
sample_color = sample_colors[sample]
smooth = use_smooth[sample]

# dz, dy, dx = 2.0, 2.0, 2.0  # nm, 2.0, 2.0, 2.0  # nm,
# voxel_nm = 2.0
# if sample == 3: 
#     crop_size = 400
# else: 
#     crop_size = 40 #200

# dz, dy, dx = 8.0, 8.0, 8.0  # nm, 2.0, 2.0, 2.0  # nm,
# voxel_nm = 8.0
# crop_size = 5


# dz, dy, dx = 8/np.sqrt(2), 8/np.sqrt(2), 8/np.sqrt(2)  # nm, 2.0, 2.0, 2.0  # nm,
# voxel_nm = dz
# if sample == 3: 
#     crop_size = 400
# else: 
#     crop_size = 40 #200


data_dir = f'C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/'

volume, fields, metadata = load_volume_analysis(data_dir, sample_name, date, smooth)

jacobian = fields['jacobian']
curl = fields['curl']
# divergence = fields['divergence']
# energies = fields['energies']
# topology = fields['Nsk']
# curvature = fields['curvature']

print("Loaded fields:", list(fields.keys()))
print("Volume shape:", volume['shape'])
print("Volume size:", 16*404*404)
print(volume.keys())



# crop volume to get rid of edge effects and apply Mz smooth if needed

In [ ]:
# crop = False
# #crop_size = 400

sigma = False

# print(crop_size)
# cropped_xy = vis.crop_xy_edges(volume, n_pix=crop_size)
# print(cropped_xy['shape'])   # (16, 304, 304)

# if crop == True: 
#     print('hi')
#     volume = cropped_xy


# # sigmas_to_test = [0, 1]

# # for s in sigmas_to_test:
# #     vis.apply_gaussian_blur_and_view(volume, field='Mz', sigma=s)

# # sigma_save = 0

vis.plot_three_slice_fancy(volume, field='Mz')

# Marching cubes isosurface extraction at Mz = 0

In [ ]:
# %% [Cell 5: Interpolate Mx, My, Mz onto mesh vertices]
def sample_volume_at_points(volume, points_idx):
    """
    Trilinear interpolation of `volume` at fractional (z, y, x) index
    coordinates given in points_idx, shape (N, 3).
    """
    coords = points_idx.T  # map_coordinates wants shape (ndim, N)
    return map_coordinates(volume, coords, order=1, mode='nearest')

## Creat Iso surface

In [ ]:
if sigma == True: 
    Mz = volume['smoothed']['Mz'][sigma_save]
else: 
    Mz =  volume['Mz']

# Mz = Mz_smooth       # shape (nz, ny, nx)
Mx = volume['Mx']
My = volume['My']

# spacing here scales the output vertex coordinates and normals into physical
# Level = 0 because this is where the domain wall center is!!! 
verts_phys, faces, normals_phys, values = marching_cubes(Mz, level=0.0, spacing=(dz, dy, dx), allow_degenerate=True) # Using allow_degenerate=False didn't visually change anything for 452

print(f"Extracted mesh: {verts_phys.shape[0]} vertices, {faces.shape[0]} triangles")

# Convert physical-space vertex coordinates back into fractional voxel-index
# coordinates (z_idx, y_idx, x_idx), since map_coordinates() needs index space.
verts_idx = verts_phys / np.array([dz, dy, dx])

#Correct normal sign convention]
###########  We want normals to consistently point from -Mz domain into +Mz domain.
# Check by sampling Mz slightly ahead of / behind each vertex along its normal.

eps_phys = dx  # step size in physical units; scale relative to your voxel size 
eps_idx = eps_phys / np.array([dz, dy, dx])

# normals_phys is in (z, y, x) order to match marching_cubes' internal axis order
step = normals_phys * eps_idx  # broadcast scale per-axis

pts_forward = verts_idx + step
pts_backward = verts_idx - step

Mz_forward = sample_volume_at_points(Mz, pts_forward)
Mz_backward = sample_volume_at_points(Mz, pts_backward)

# define chirality at +Mz to -Mz
flip_mask = Mz_forward > Mz_backward
normals_phys[flip_mask] *= -1

print(f"Flipped {flip_mask.sum()} / {len(flip_mask)} normals to enforce -Mz -> +Mz convention")

# Reorder normals and vertex from (z, y, x) to (x, y, z) for consistency with M_v and PyVista
normals_xyz = normals_phys[:, ::-1]
verts_xyz = verts_phys[:, ::-1]  


## Sample Volume at points to create useful mesh

In [ ]:
Mx_v = sample_volume_at_points(Mx, verts_idx)
My_v = sample_volume_at_points(My, verts_idx)
Mz_v = sample_volume_at_points(Mz, verts_idx)  # sample raw (unsmoothed) Mz for the actual physical vector
M_v = np.stack([Mx_v, My_v, Mz_v], axis=1)      # shape (N, 3), order (x, y, z) per vertex

## Calculate dw angles

In [ ]:
def compute_chirality_angle(M_v, normals_xyz, up_axis=np.array([0.0, 0.0, 1.0])):
    """
    Compute a signed angle capturing both wall type (Neel vs Bloch) and
    chirality (rotation sense), for each mesh vertex.

    Parameters
    ----------
    M_v : (N, 3) array
        Magnetization vectors at each mesh vertex (x, y, z order).
    normals_xyz : (N, 3) array
        Wall normal vectors at each mesh vertex (x, y, z order), already
        sign-corrected to point from -Mz into +Mz domain.
    up_axis : (3,) array
        Reference axis used to define tangential direction, defaults to
        the film's easy axis (z), appropriate for PMA films.

    Returns
    -------
    psi_chiral : (N,) array, degrees, range (-180, 180]
        0 deg   -> Neel, pointing along +n_hat
        180 deg -> Neel, pointing along -n_hat (opposite polarity)
        +90 deg -> Bloch, curling in +t_hat direction
        -90 deg -> Bloch, curling in -t_hat direction
    t_hat : (N, 3) array
        The tangential unit vector used at each vertex, in case you want
        to visualize it directly (e.g. as arrows alongside the normals).
    degenerate_mask : (N,) bool array
        True where n_hat was parallel to up_axis, making t_hat undefined
        (chirality is meaningless there; check how many/where this occurs).
    """
    up = np.broadcast_to(up_axis, normals_xyz.shape)
    t_vec = np.cross(up, normals_xyz)  # automatically perpendicular to normals_xyz
    t_norm = np.linalg.norm(t_vec, axis=1)

    degenerate_mask = t_norm < 1e-8
    if np.any(degenerate_mask):
        print(f"Warning: {degenerate_mask.sum()} / {len(degenerate_mask)} vertices "
              f"have normals parallel to up_axis; chirality undefined there.")
        t_norm = np.where(degenerate_mask, 1.0, t_norm)  # avoid divide-by-zero

    t_hat = t_vec / t_norm[:, None]

    M_n = np.einsum('ij,ij->i', M_v, normals_xyz)   # signed, along normal
    M_t_signed = np.einsum('ij,ij->i', M_v, t_hat)   # signed, along tangent

    psi_chiral = np.degrees(np.arctan2(M_t_signed, M_n))

    return psi_chiral, t_hat, degenerate_mask

In [ ]:
# Compute Bloch/Néel character angle at each vertex]
M_n = np.einsum('ij,ij->i', M_v, normals_xyz)          # M component along normal
#M_n = np.linalg.norm(M_v, axis=1)
M_t_vec = M_v - M_n[:, None] * normals_xyz
M_t_mag = np.linalg.norm(M_t_vec, axis=1)
psi_unsigned = np.degrees(np.arctan2(M_t_mag, M_n))             # 0/180 = Néel-like, +-90 = Bloch-like
psi_chiral, t_hat, degenerate_mask = compute_chirality_angle(M_v, normals_xyz)


# Normalize M_n for future calculations. Allows the assignment of Neel and Bloch to be more robust. 
M_mag = np.linalg.norm(M_v, axis=1)               # local |M| at each vertex

# guard against divide-by-zero where |M| is ~0
eps = 1e-12
M_n_n = np.where(M_mag > eps, M_n / np.maximum(M_mag, eps), np.nan)

In [ ]:
# div_x = sample_volume_at_points(divergence['div_x'], verts_idx)
# div_y = sample_volume_at_points(divergence['div_y'], verts_idx)
# div_z = sample_volume_at_points(divergence['div_z'], verts_idx)  # sample raw (unsmoothed) Mz for the actual physical vector
# div_M= sample_volume_at_points(divergence['div_M'], verts_idx) 
# source = sample_volume_at_points(divergence['source_mask'], verts_idx) 
# sink = sample_volume_at_points(divergence['sink_mask'], verts_idx) 

# kappa = sample_volume_at_points(curvature['kappa_map'], verts_idx)


Convert Kappa to a sample-able volume

In [ ]:
# n_layers = 16
# Kappa_vol = np.repeat(curvature['kappa_map'][np.newaxis, :, :], n_layers, axis=0)   # shape (16, 404, 404)


# valid = ~np.isnan(vol)
# filled = np.where(valid, vol, 0.0)

# # coords: shape (3, N), in the same index order as vol
# num = map_coordinates(filled, coords, order=1, mode='nearest')
# den = map_coordinates(valid.astype(float), coords, order=1, mode='nearest')
# sampled = np.where(den > 0, num / den, np.nan)

## save iso cut in df

In [ ]:
def build_wall_dataframe(verts_xyz, normals_xyz, t_hat, M_v,
                          psi_unsigned, psi_chiral, M_normal, degenerate_mask,
                          spacing=(2.0, 2.0, 2.0)):
    """
    Consolidate all per-vertex wall quantities into a single tidy DataFrame,
    one row per mesh vertex. spacing = (dz, dy, dx) in physical units, used
    to also store voxel-index coordinates for binning/slicing by layer.
    """
    dz, dy, dx = spacing
    df = pd.DataFrame({
        'x': verts_xyz[:, 0], 'y': verts_xyz[:, 1], 'z': verts_xyz[:, 2],
        'ix': verts_xyz[:, 0] / dx, 'iy': verts_xyz[:, 1] / dy, 'iz': verts_xyz[:, 2] / dz,
        'nx': normals_xyz[:, 0], 'ny': normals_xyz[:, 1], 'nz': normals_xyz[:, 2],
        'tx': t_hat[:, 0], 'ty': t_hat[:, 1], 'tz': t_hat[:, 2],
        'Mx': M_v[:, 0], 'My': M_v[:, 1], 'Mz': M_v[:, 2],
        'M_normal': M_n,
        'psi_unsigned': psi_unsigned,
        'psi_chiral': psi_chiral,
        'degenerate': degenerate_mask,
    })
    df['vertex_id'] = df.index
    return df

wall_df = build_wall_dataframe(verts_xyz, normals_xyz, t_hat, M_v,
                                psi_unsigned, psi_chiral, M_n, degenerate_mask, spacing=(dz, dy, dx))

In [ ]:
# wall_df['curl_magnitude'] = sample_volume_at_points(curl['curl_magnitude'], verts_idx)
# wall_df['PMA'] = sample_volume_at_points(energies['PMA'], verts_idx)
# wall_df['Exchange'] = sample_volume_at_points(energies['Divergence'], verts_idx)
# wall_df['iDMI'] = sample_volume_at_points(energies['iDMI'], verts_idx)
# wall_df['Nsk'] = sample_volume_at_points(topology['q_density'], verts_idx)

wall_df['Mx'] = Mx_v
wall_df['My'] = My_v

# wall_df['div_M'] = div_M
# wall_df['source'] = source
# wall_df['sink'] = sink

# wall_df['kappa'] = kappa

wall_df['M_n_n'] = M_n_n


In [ ]:
assert np.allclose(wall_df['M_n_n'], np.cos(np.radians(wall_df['psi_unsigned'])), equal_nan=True)

# Plot Iso-Surface

In [ ]:
def add_volume_slice_pv(plotter, sample, field='Mz', axis='z', index=None,
                         sigma=None, spacing=(2.0, 2.0, 2.0),
                         cmap='viridis', opacity=0.7, vmin=None, vmax=None,
                         show_scalar_bar=False):
    """
    Add a single axis-aligned voxel slice of `field` to a PyVista plotter,
    using the same index convention as plot_three_slice (axis + voxel index).

    axis : 'x', 'y', or 'z' -- which axis to slice along
    index : voxel index along that axis (defaults to center)
    sigma : if not None, use the cached smoothed volume at that sigma
            (sample['smoothed'][key][sigma], as populated by plot_three_slice)
    spacing : (dz, dy, dx) physical voxel size, e.g. (8.0, 8.0, 8.0) for your data
    """
    field_aliases = {'mag': 'magnitude', 'magnitude': 'magnitude',
                      'mx': 'Mx', 'my': 'My', 'mz': 'Mz'}
    key = field_aliases.get(field.lower())
    data = sample[key] if sigma is None else sample['smoothed'][key][sigma]

    nz, ny, nx = sample['shape']
    dz, dy, dx = spacing

    if vmin is None:
        vmin = np.nanmin(sample[key])
    if vmax is None:
        vmax = np.nanmax(sample[key])

    # Build the full 3D grid with correctly-ordered cell data
    grid = pv.ImageData()
    grid.dimensions = np.array((nx, ny, nz)) + 1  # point dims = cell dims + 1
    grid.spacing = (dx, dy, dz)
    grid.origin = (0, 0, 0)
    grid.cell_data[key] = data.flatten(order='C')  # corrected order

    # Default index = center of chosen axis, same as plot_three_slice
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    if index is None:
        index = default_index[axis]

    normal_map = {'x': (1, 0, 0), 'y': (0, 1, 0), 'z': (0, 0, 1)}
    spacing_map = {'x': dx, 'y': dy, 'z': dz}
    axis_pos_idx = {'x': 0, 'y': 1, 'z': 2}

    # Place the cutting plane at the *center* of the chosen voxel layer,
    # so the cut passes through the middle of those cells and returns their
    # exact values rather than something interpolated between layers.
    origin = [0.0, 0.0, 0.0]
    origin[axis_pos_idx[axis]] = (index + 0.5) * spacing_map[axis]

    sliced = grid.slice(normal=normal_map[axis], origin=tuple(origin))

    plotter.add_mesh(sliced, scalars=key, cmap=cmap, opacity=opacity,
                      clim=[vmin, vmax], show_scalar_bar=show_scalar_bar)
    return sliced

def plot_map(sample, verts_xyz, faces, plot_dict, sargs, voxel_nm = 2.0, show_grid = True, field = 'Mz', axis = 'z'):
    """
    Visualize the signed chirality/wall-type angle on the domain wall mesh.

    Parameters
    ----------
    verts_xyz, faces : mesh geometry, as produced by marching_cubes
    psi_chiral : (N,) array, degrees, from compute_chirality_angle
    cmap : colormap; 'twilight' or 'hsv' are good choices since they're
        cyclic and represent the wraparound at +-180 deg without a
        visually misleading hard color jump
    show_normals, show_tangents : optionally overlay arrows for n_hat
        and/or t_hat (pass normals_xyz / t_hat if enabling these)
    """

    axis_view = axis
    field_view = field

    cmap = plot_dict['color_map']
    min = plot_dict['bounds'][0]
    max = plot_dict['bounds'][1]


    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
    mesh = pv.PolyData(verts_xyz, faces_pv)
    mesh['key_to_map'] = plot_dict['view']

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh, scalars='key_to_map', cmap=cmap, clim=[min, max], scalar_bar_args=sargs)

    if show_grid: 
        M = sample[field]
        grid = pv.ImageData()
        grid.dimensions = np.array(Mz.shape[::-1]) + 1   # (Mz = samples[sample_name]['cropped']['Mz'] )
        # print(np.array(Mz.shape[::-1]) +1 )
        grid.spacing = (voxel_nm, voxel_nm, voxel_nm) #################### HARD CODED FOR 8nm VOXEL SIZE 
        grid.origin = (0, 0, 0)
        grid.cell_data['Mz'] = Mz.flatten(order='C')  # note: flatten order matters, check orientation
        add_volume_slice_pv(plotter, sample, field=field_view, axis=axis_view, index=0, spacing=(voxel_nm, voxel_nm, voxel_nm), opacity=0.2)  # defaults to center


    plotter.view_isometric()      # view_isometric   view_xy
    plotter.camera.zoom(plot_dict['zoom'])
    plotter.camera.azimuth += 270
    plotter.add_axes()
    plotter.show()
    
    return mesh

In [ ]:
# mesh['Normals'] = normals_xyz
# mesh['Mn'] = M_n
# mesh['psi_deg'] = psi_unsigned
# mesh['Mz_at_vertex'] = Mz_v
# mesh['Mx'] = Mx_v
# mesh['My'] = My_v

## Computed Angles

In [ ]:
# # # # # Wall surface colored by psi character
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n psi (deg): 0/180=Neel, +-90=Bloch'
# )

# psi_plot_dict = dict(
#     view= psi_unsigned, 
#     bounds = [0, 90],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Surface Normal 0=Bloch, +-Mn=Neel'
)

psi_plot_dict = dict(
    view= M_n, 
    bounds = [np.min(My_v), np.max(My_v)],
    color_map = 'brg', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, voxel_nm = voxel_nm,  show_grid = True, field = 'Mz', axis = 'z')


sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Surface Normal 0=Bloch, +-1=Neel'
)

psi_plot_dict = dict(
    view= M_n_n, 
    bounds = [-1, 1],
    color_map = 'brg', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, voxel_nm = voxel_nm,  show_grid = True, field = 'Mz', axis = 'z')


In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Psi Chiral '
# )

# psi_plot_dict = dict(
#     view= psi_chiral, 
#     bounds = [-180, 180],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')


## Computed Vectors and Energies

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Kappa'
# )

# psi_plot_dict = dict(
#     view= wall_df['kappa'].values, 
#     bounds = [-0.001, 0.001],
#     color_map = 'coolwarm', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Divergence'
# )

# psi_plot_dict = dict(
#     view= wall_df['Exchange'].values *10000000, 
#     bounds = [0, 1],
#     color_map = 'Purples', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n PMA'
# )

# psi_plot_dict = dict(
#     view= wall_df['PMA'].values *1000000 , 
#     bounds = [0, 1],
#     color_map = 'Oranges', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Curl'
# )

# psi_plot_dict = dict(
#     view= wall_df['curl_magnitude'].values *1000 , 
#     bounds = [-0.1, 0.1],
#     color_map = 'RdBu_r', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n Divergence'
# )

# psi_plot_dict = dict(
#     view= wall_df['div_M'].values *10000 , 
#     bounds = [-0.1, 0.1],
#     color_map = 'plasma', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n source'
# )

# psi_plot_dict = dict(
#     view= wall_df['source'].values, 
#     bounds = [-1, 1],
#     color_map = 'Greys', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n sink'
# )

# psi_plot_dict = dict(
#     view= wall_df['sink'].values, 
#     bounds = [-1, 1],
#     color_map = 'Greys', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n  iDMI'
# )

# psi_plot_dict = dict(
#     view= wall_df['iDMI'].values * 10, 
#     bounds = [-0.5, 0.5],
#     color_map = 'Greens', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

In [ ]:
# sargs = dict(
#     position_x=0.4,  # Move towards the right edge
#     position_y=0.85,   # Move near the bottom
#     height=0.05,       # Make it take up 50% of window height
#     width=0.5,        # Width of the bar
#     vertical=False,    # Orientation
#     title = f'{sample_name} \n  Nsk'
# )

# psi_plot_dict = dict(
#     view= wall_df['Nsk'].values*1000, 
#     bounds = [-1, 1],
#     color_map = 'seismic', 
#     zoom = 1
# )

# plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

# Analyze ISO-surface

In [ ]:
## Geometry and statistics functions
def film_geometry(wall_df, voxel_nm=2.0, ):
    """
   
    """
    z_min = wall_df['z'].min() 
    z_max = wall_df['z'].max() 

    z_center = 0.5 * (z_min + z_max)

    thickness_nm = (z_max - z_min) + voxel_nm ### not sure if this should be interpolated or voxel
    layers = np.arange(int(round(z_min / voxel_nm)), int(round(z_max / voxel_nm)) + 1)

    geom = dict(z_min=z_min, z_max=z_max, z_center=z_center,
                thickness=thickness_nm,
                z_bottom=z_center - thickness_nm / 2, z_top=z_center + thickness_nm / 2,
                voxel_nm=voxel_nm, layers=layers, x_vals=layers * voxel_nm - z_center)
    print(f"film: vertices span {z_min:.1f} to {z_max:.1f} nm, {len(layers)} layers, "
          f"thickness {thickness_nm:.1f} nm, center {z_center:.1f} nm")
    return geom



def _binned_raw_stats(bin_idx, z, w, cl, n_bins, k=None):
    """
    Non-bootstrap companion to _binned_cluster_stats, for checking its assumptions.

    bin_idx : int bin index per point; z : value; w : weight; cl : compact int cluster id
    k       : optional raw kappa per point; if given, also fits a weighted linear trend z ~ k

    Returns mean, err, n_pts, n_clusters, extra
      mean : weighted mean per bin (should match the bootstrap point estimate exactly)
      err  : naive SEM assuming independent points (std / sqrt(n_eff))
      extra (dict):
        std         : raw (weighted) standard deviation of z within each bin
        n_eff       : Kish effective sample size, (sum w)^2 / sum w^2
        sem_cluster : analytic cluster-robust SEM (ratio-estimator linearization),
                      the no-resampling analogue of the bootstrap err
        deff        : (sem_cluster / err)^2, design effect; ~1 means points act independent,
                      >>1 means within-cluster correlation is inflating the true error
        trend       : dict(slope, intercept, se_naive, se_cluster) if k is given
    """
    bin_idx = np.asarray(bin_idx)
    n_cl = cl.max() + 1

    # point-level per-bin sums
    Sw = np.bincount(bin_idx, weights=w, minlength=n_bins)
    Sw2 = np.bincount(bin_idx, weights=w**2, minlength=n_bins)
    Szw = np.bincount(bin_idx, weights=w * z, minlength=n_bins)
    n_pts = np.bincount(bin_idx, minlength=n_bins)

    # per-cluster, per-bin weight sums (for n_clusters and cluster-robust SE)
    Sw_cb = np.zeros((n_cl, n_bins))
    Szw_cb = np.zeros((n_cl, n_bins))
    np.add.at(Sw_cb, (cl, bin_idx), w)
    np.add.at(Szw_cb, (cl, bin_idx), w * z)
    n_clusters = (Sw_cb > 0).sum(0)

    with np.errstate(invalid='ignore', divide='ignore'):
        mean = Szw / Sw

        # two-pass weighted variance, with reliability-weight small-sample correction
        dev2 = (z - mean[bin_idx]) ** 2
        var_pop = np.bincount(bin_idx, weights=w * dev2, minlength=n_bins) / Sw
        n_eff = Sw**2 / Sw2
        std = np.sqrt(var_pop * n_eff / (n_eff - 1))
        err = std / np.sqrt(n_eff)

        # cluster-robust SEM: residual of each cluster's contribution to the ratio estimator
        resid = Szw_cb - mean * Sw_cb
        sem_cluster = np.sqrt(n_cl / (n_cl - 1) * (resid**2).sum(0)) / Sw
        deff = (sem_cluster / err) ** 2

    extra = dict(std=std, n_eff=n_eff, sem_cluster=sem_cluster, deff=deff)

    if k is not None:
        k = np.asarray(k, dtype=float)
        W = w.sum()
        kbar = (w * k).sum() / W
        zbar = (w * z).sum() / W
        kc = k - kbar
        Sxx = (w * kc**2).sum()
        slope = (w * kc * (z - zbar)).sum() / Sxx
        intercept = zbar - slope * kbar
        e = z - (intercept + slope * k)
        n = len(z)
        se_naive = np.sqrt((w * e**2).sum() / (n - 2) / Sxx)
        score_c = np.bincount(cl, weights=w * kc * e, minlength=n_cl)  # per-cluster score
        se_cluster = np.sqrt(n_cl / (n_cl - 1) * (score_c**2).sum()) / Sxx
        extra['trend'] = dict(slope=slope, intercept=intercept,
                              se_naive=se_naive, se_cluster=se_cluster)

    return mean, err, n_pts, n_clusters, extra


In [ ]:
def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

def _bimodal(x, A1, mu1, s1, A2, mu2, s2):
    return _gaussian(x, A1, mu1, s1) + _gaussian(x, A2, mu2, s2)

## Bloch vs thickness

In [ ]:
## thickness functions
def find_M0_crossings_mesh(wall_df, faces, geom, angle_col='M_n_n', curv_col='kappa',
                           voxel_nm=2.0, cluster_gap_nm=8.0, drop_degenerate=False):
    """
    Exact M_normal = 0 points along mesh edges, grouped by lateral column.
    Returns
      z0_df     : one row per column with a single crossing
                  (col_x, col_y, x, y, z0_nm, z0_rel_nm, kappa, n_cross)
      misc_df   : columns with status 'multiple' or 'none' (z0_list_rel_nm for multiple)
      crossings : every edge crossing (x, y, z, z_rel_nm, kappa, col_x, col_y, status)
    z0_rel_nm is relative to the film center from geom, like the other plots.
    """
    assert (wall_df.index.to_numpy() == np.arange(len(wall_df))).all(), \
        "wall_df index must be 0..N-1 to match face indices (use reset_index(drop=True))"
    faces = np.asarray(faces, dtype=int)
    # Points
    P = wall_df[['x', 'y', 'z']].to_numpy(float)
    # angle definition
    m = wall_df[angle_col].to_numpy(float).copy()
    # kappa
    k = wall_df[curv_col].to_numpy(float)
    if drop_degenerate and 'degenerate' in wall_df.columns:
        m[wall_df['degenerate'].to_numpy(bool)] = np.nan

    m[m == 0] = 1e-12                      # exact zeros count as positive

    # unique mesh edges whose endpoints have opposite sign
    edges = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]])
    edges = np.unique(np.sort(edges, axis=1), axis=0)
    a, b = edges[:, 0], edges[:, 1]
    with np.errstate(invalid='ignore'):
        cross = (m[a] * m[b]) < 0          # walls with a crossing will have the angle at the edges less than 0 (ake the wall  must go from 1 to -1 so a 0 position exists)
    a, b = a[cross], b[cross]
    t = m[a] / (m[a] - m[b])               # fraction of the way from a to b
    pts = P[a] + t[:, None] * (P[b] - P[a])
    kap = k[a] + t * (k[b] - k[a])         # NaN if either end has no kappa

    lo, hi = P.min(axis=0), P.max(axis=0)
    vert_ok = np.ones(len(P), bool)

    zc = geom['z_center']
    cr = pd.DataFrame({'x': pts[:, 0], 'y': pts[:, 1], 'z': pts[:, 2],
                       'z_rel_nm': pts[:, 2] - zc, 'kappa': kap,
                       'col_x': np.rint(pts[:, 0] / voxel_nm).astype(int),
                       'col_y': np.rint(pts[:, 1] / voxel_nm).astype(int)})

    singles, multis = [], []
    for (cx, cy), g in cr.groupby(['col_x', 'col_y']):
        z = np.sort(g['z'].to_numpy())
        groups = np.split(z, np.where(np.diff(z) > cluster_gap_nm)[0] + 1)
        if len(groups) == 1:
            singles.append({'col_x': cx, 'col_y': cy, 'x': g['x'].mean(), 'y': g['y'].mean(),
                            'z0_nm': z.mean(), 'z0_rel_nm': z.mean() - zc,
                            'kappa': g['kappa'].mean(), 'n_cross': len(z)})
        else:
            multis.append({'col_x': cx, 'col_y': cy, 'status': 'multiple',
                           'z0_list_rel_nm': [gz.mean() - zc for gz in groups]})

    z0_df = pd.DataFrame(singles, columns=['col_x', 'col_y', 'x', 'y', 'z0_nm', 'z0_rel_nm',
                                           'kappa', 'n_cross'])
    multi_df = pd.DataFrame(multis, columns=['col_x', 'col_y', 'status', 'z0_list_rel_nm'])

    # columns that contain wall vertices but no crossing
    vcols = pd.DataFrame({'col_x': np.rint(P[vert_ok, 0] / voxel_nm).astype(int),
                          'col_y': np.rint(P[vert_ok, 1] / voxel_nm).astype(int)}).drop_duplicates()
    have = pd.concat([z0_df[['col_x', 'col_y']], multi_df[['col_x', 'col_y']]])
    none_df = vcols.merge(have, on=['col_x', 'col_y'], how='left', indicator=True)
    none_df = none_df[none_df['_merge'] == 'left_only'][['col_x', 'col_y']].copy()
    none_df['status'] = 'none'
    none_df['z0_list_rel_nm'] = [[] for _ in range(len(none_df))]
    misc_df = pd.concat([multi_df, none_df], ignore_index=True)

    multi_keys = set(zip(multi_df['col_x'], multi_df['col_y']))
    cr['status'] = np.where([(i, j) in multi_keys for i, j in zip(cr['col_x'], cr['col_y'])],
                            'multiple', 'single')

    n_tot = len(z0_df) + len(misc_df)
    print(f"{len(cr)} edge crossings; {n_tot} columns total")
    print(f"  single   : {len(z0_df)} ({len(z0_df) / n_tot:.0%})")
    print(f"  multiple : {len(multi_df)} ({len(multi_df) / n_tot:.0%})")
    print(f"  none     : {len(none_df)} ({len(none_df) / n_tot:.0%})  "
          f"(approximate: the crossing may fall in a neighboring column)")
    print(f"  kappa valid for {z0_df['kappa'].notna().sum()} of the single columns")
    return z0_df, misc_df, cr

def plot_Bloch_angle_vs_z_m0(wall_df, faces, geom=None, angle_col='M_n_n', curv_col='kappa',
                             color='green', title=None, voxel_nm=8.0, bin_nm=None,
                             cluster_gap_nm=8.0, drop_degenerate=True,
                             show_fit=True, ylim=(-1.1, 1.1), figsize=(24, 7)):
    """
    bin_nm : z bin size (nm) used for the profile, the crossing counts and the Gaussian fit.
             Defaults to voxel_nm. Set it to the real resolution, not the interpolated voxel size.
    """
    if bin_nm is None:
        bin_nm = voxel_nm
    if geom is None:
        geom = film_geometry(wall_df, voxel_nm)
    layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']

    wdf = wall_df if curv_col in wall_df.columns else wall_df.assign(**{curv_col: np.nan})
    z0_df, misc_df, crossings = find_M0_crossings_mesh(
        wdf, faces, geom, angle_col=angle_col, curv_col=curv_col, voxel_nm=voxel_nm,
        cluster_gap_nm=cluster_gap_nm, drop_degenerate=drop_degenerate)
    z0_df['z0_layer'] = np.rint(z0_df['z0_nm'] / voxel_nm).astype(int)

    # ---- per-layer profile (fine, interpolated grid): this is what gets returned ----
    d = wall_df[['z', angle_col]].copy()


    z_min, z_max = d['z'].min(), d['z'].max()
    z_center = 0.5 * (z_min + z_max)                    # film center (symmetric crop)
    T = (z_max - z_min) + voxel_nm                      # film thickness covered by the samples
    n_bins = int(round(T / bin_nm))

    if drop_degenerate and 'degenerate' in wall_df.columns:
        d = d[~wall_df['degenerate'].astype(bool)]
    d = d.dropna(subset=[angle_col])
    # d['z_layer'] = np.rint(d['z'] / voxel_nm).astype(int)
    # grouped = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(layers)
    # grouped['z_rel_nm'] = x_vals

    d['z_layer'] = np.rint((d['z'] - z_min) / voxel_nm).astype(int)    # 0, 1, 2, ...
    layers = np.arange(d['z_layer'].max() + 1)
    x_vals = z_min + layers * voxel_nm - z_center                      # layer centers relative to film center
    grouped_f = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(layers)
    grouped_f['z_rel_nm'] = x_vals

    # ---- coarse z bins at the real resolution, centered on the film center ----
    # ext = max(abs(x_vals.min()), abs(x_vals.max())) + voxel_nm / 2
    # n_half = int(np.ceil((ext - bin_nm / 2) / bin_nm))
    # zedges = (np.arange(-n_half, n_half + 2) - 0.5) * bin_nm      # one bin centered on z = 0
    # centers = 0.5 * (zedges[:-1] + zedges[1:])
    # nb = len(centers)
    # width = 0.9 * bin_nm
    # print(f"{nb} z bins of {bin_nm:g} nm (interpolated voxel = {voxel_nm:g} nm)")

    zedges = (np.arange(n_bins + 1) - n_bins / 2) * bin_nm             # e.g. -16, -8, 0, 8, 16
    centers = 0.5 * (zedges[:-1] + zedges[1:])                         # e.g. -12, -4, 4, 12
    nb = len(centers)
    width = 0.9 * bin_nm
    print(f"{nb} z bins of {bin_nm:g} nm, centers at {centers}")

    # profile of angle_col on the coarse bins
    d['z_rel'] = d['z'] - z_center
    d['bin'] = np.digitize(d['z_rel'], zedges) - 1
    d = d[(d['bin'] >= 0) & (d['bin'] < nb)]
    grouped_b = d.groupby('bin')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(range(nb))

    # ---- crossing counts per coarse bin ----
    y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)
    n_out = len(z0_df) - y.sum()
    if n_out:
        print(f"{n_out} crossings fall outside the film layers and are not shown")

    # ---- statistics from the individual z0 values (independent of bin size) ----
    z0 = z0_df['z0_rel_nm'].to_numpy()
    z0 = z0[(z0 >= zedges[0]) & (z0 <= zedges[-1])]
    stats = {'n': len(z0)}
    if len(z0) > 1:
        stats.update(mean=float(np.mean(z0)), median=float(np.median(z0)),
                     variance=float(np.var(z0, ddof=1)), std=float(np.std(z0, ddof=1)), sterr = (float(np.std(z0, ddof=1)) / np.sqrt(len(z0))))
    else:
        stats = None

    # ---- Gaussian fit to the binned counts ----
    A = mu = sigma = fwhm = r2 = None
    if show_fit and stats is not None and y.sum() > 0 and nb >= 4:
        try:
            p0 = [y.max(), stats['mean'], max(stats['std'], bin_nm)]
            (A, mu, sigma), _ = curve_fit(_gaussian, centers, y, p0=p0, maxfev=10000)
            sigma = abs(sigma)
            fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
            resid = y - _gaussian(centers, A, mu, sigma)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else np.nan
            stats.update(A=float(A), mu=float(mu), sigma=float(sigma),
                         fwhm=float(fwhm), r2=float(r2))
        except RuntimeError:
            print("Gaussian fit did not converge; plotting without it")
            show_fit = False
    else:
        show_fit = False

    if stats is not None:
        print(f"z0 (nm from center): n = {stats['n']}, mean = {stats['mean']:.2f}, "
              f"median = {stats['median']:.2f}, variance = {stats['variance']:.2f} nm^2")
        if show_fit:
            print(f"Gaussian fit: mu = {mu:.2f} nm, sigma = {sigma:.2f} nm, "
                  f"FWHM = {fwhm:.2f} nm, R² = {r2:.2f}")

    # ---- plots ----
    xlabel = 'z relative to film center (nm)'
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 2, width_ratios=[1, 1], figure=fig)
    ax2 = fig.add_subplot(gs[0])
    ax3 = fig.add_subplot(gs[1], sharex=ax2)

    # Panel 1: profile on the coarse bins
    ax2.errorbar(centers, grouped_b['mean'], yerr=grouped_b['sem'], xerr = stats['sterr'], fmt='o-',
                 capsize=3, color=color)
    ax2.axhline(0, color='magenta', ls='--', label='Bloch (0)')
    ax2.axhline(1, color='cyan', ls='--', label='Neel (±1)')
    ax2.axhline(-1, color='cyan', ls='--')
    ax2.axvline(0, color='k', ls='--', label='Film center')
    if ylim is not None:
        ax2.set_ylim(*ylim)
    if stats:
        ax2.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} nm")
        ax2.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
    ax2.set_xlabel(xlabel, fontsize=25)
    ax2.set_ylabel(angle_col, fontsize=25)
    ax2.legend(fontsize=20)

    # Panel 2: counts where M_norm = 0
    ax3.bar(centers, y, width=width, color=color, alpha=0.5, edgecolor='gray')
    ax3.axvline(0, color='k', ls='--', label='Film center')
    if stats:
        ax3.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} nm")
        ax3.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
        ax3.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm^2")
    if show_fit:
        xs = np.linspace(zedges[0], zedges[-1], 300)
        ax3.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
                 label=f"Gaussian fit (R² = {r2:.2f})")
        ax3.hlines(A / 2, mu - fwhm / 2, mu + fwhm / 2, color='k', lw=2.5, alpha=0,
                   label=f"FWHM = {fwhm:.2f} nm")
        ax3.axvline(mu, color='darkorange', lw=1, ls=':')

    ax3.legend(loc='upper left', fontsize=20)
    ax3.set_xlabel(xlabel, fontsize=25)
    ax3.set_ylabel(f'# of Bloch points per {bin_nm:g} nm', fontsize=25)
    ax3.set_xticks(centers)
    ax3.set_xticklabels([f'{v:g}' for v in centers], rotation=45)

    fig.suptitle(f'{angle_col} vs. thickness position ({bin_nm:g} nm bins) \n {title}', fontsize=30)
    plt.tight_layout()
    plt.show()

    return grouped_b, z0_df, misc_df, crossings, stats


def plot_z0_on_isosurface(wall_df, faces, crossings, z0_df, voxel_nm=2.0,
                          window=None, z_scale=1.0, m_clim=None, z0_clim=4.0, point_size=6):
    """
    Two linked PyVista panels of the isosurface:
      left : surface colored by M_normal; black dots = single-column crossings,
             magenta = crossings in columns dropped as 'multiple'. Dots should sit on the
             pale (M_normal ~ 0) band.
      right: surface colored by the z0 assigned to each vertex's lateral column
             (nm from center; gray = column has no single z0), black dots = crossings.
    window  : (xmin, xmax, ymin, ymax) in nm. Strongly recommended: the film is ~30 nm
              thick and ~700 nm wide, so zoom to something like 150 x 150 nm.
    z_scale : vertical exaggeration (the film is thin, so z0 differences are invisible at 1).
    m_clim  : half-range of the M_normal color scale (None = 95th percentile of |M_normal|).
    """
    P = wall_df[['x', 'y', 'z']].to_numpy(float)
    faces = np.asarray(faces, dtype=int)
    cr = crossings
    if window is not None:
        xmin, xmax, ymin, ymax = window
        inwin = (P[:, 0] >= xmin) & (P[:, 0] <= xmax) & (P[:, 1] >= ymin) & (P[:, 1] <= ymax)
        faces = faces[inwin[faces].all(axis=1)]
        cr = cr[(cr['x'] >= xmin) & (cr['x'] <= xmax) & (cr['y'] >= ymin) & (cr['y'] <= ymax)]

    mesh = pv.PolyData(P, np.hstack([np.full((len(faces), 1), 3), faces]).ravel())
    Mn = wall_df['M_n_n'].to_numpy(float)
    mesh.point_data['M_n_n'] = Mn
    key = pd.MultiIndex.from_arrays([np.rint(P[:, 0] / voxel_nm).astype(int),
                                     np.rint(P[:, 1] / voxel_nm).astype(int)])
    lut = z0_df.set_index(['col_x', 'col_y'])['z0_rel_nm']
    mesh.point_data['z0_rel_nm'] = lut.reindex(key).to_numpy()

    if m_clim is None:
        m_clim = np.nanpercentile(np.abs(Mn), 95)
    single = cr[cr['status'] == 'single']
    multi = cr[cr['status'] == 'multiple']

    pl = pv.Plotter()

    pl.add_mesh(mesh, scalars='M_n_n', cmap='coolwarm', clim=(-m_clim, m_clim),
                nan_color='lightgray', scalar_bar_args={'title': 'M_n_n'})
    if len(single):
        pl.add_points(single[['x', 'y', 'z']].to_numpy(), color='magenta',
                      point_size=point_size, render_points_as_spheres=True)
    if len(multi):
        pl.add_points(multi[['x', 'y', 'z']].to_numpy(), color='magenta',
                      point_size=point_size, render_points_as_spheres=True)

    pl.set_scale(zscale=z_scale)

    pl.show()



def simple_scatter_plot (wall_df, angle_col = 'M_n_n', color = 'green', voxel_nm = 8.0):


    d = wall_df[['z', angle_col]].copy()

    # ---- plot ----
    fig, ax = plt.subplots(1,1, figsize=(5,5))

    # ---- 2. Scatter plot: z vs. kappa ----

    ax.scatter(d['z'], d[angle_col], c=color,  s=15, alpha=0.6)   # <-- change 'x' to your column
    ax.axhline(np.mean(d[angle_col]), color='red', ls=':', label = f'mean = {np.mean(d[angle_col]):.2f}')
    ax.axhline(np.median(d[angle_col]), color='blue', ls=':', label = f'median = {np.median(d[angle_col]):.2f}')

    ax.set_xlabel('z layer (nm)', fontsize=14)
    ax.set_ylabel('M_normal', fontsize=14)
    ax.set_title(f'z vs. {angle_col}')
    # ax[1,0].set_ylim(np.min(xs), np.max(xs) )

    ax.legend(loc = 'upper left')

    plt.tight_layout()
    plt.show()



In [ ]:
geom = film_geometry(wall_df, voxel_nm=voxel_nm)

simple_scatter_plot(wall_df, angle_col = 'M_n_n', color = sample_color, voxel_nm = voxel_nm)

grouped, z0_df, misc_df, crossings, stats = plot_Bloch_angle_vs_z_m0(
    wall_df, faces, geom=geom, voxel_nm=voxel_nm, bin_nm=voxel_nm, drop_degenerate=False, color=sample_color, title=f'{sample_name}')

plot_z0_on_isosurface(wall_df, faces, crossings, z0_df, voxel_nm=voxel_nm)

In [ ]:
#stop

## Bloch vs curvature

### compute Kappa on isosurface centerline

In [ ]:
def vertex_areas(verts_xyz, faces):
    """Area of mesh attributed to each vertex (1/3 of each adjacent triangle)."""
    v = np.asarray(verts_xyz, float)
    tri = v[faces]                                    # (M, 3, 3)
    tri_area = 0.5 * np.linalg.norm(
        np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
    area = np.zeros(len(v))
    for k in range(3):
        np.add.at(area, faces[:, k], tri_area / 3.0)
    return area


wall_df['v_area'] = vertex_areas(verts_xyz, faces)

In [ ]:
def _d_ds(f, ds, closed):
    """Central-difference derivative along the contour (wraps around for closed loops)."""
    if closed:
        return (np.roll(f, -1) - np.roll(f, 1)) / (2 * ds)
    return np.gradient(f, ds)

def _signed_curvature(x, y, ds, closed):
    xp, yp = _d_ds(x, ds, closed), _d_ds(y, ds, closed)
    xpp, ypp = _d_ds(xp, ds, closed), _d_ds(yp, ds, closed)
    return (xp * ypp - yp * xpp) / (xp**2 + yp**2) ** 1.5


# ---------------- function 1: Mz -> curvature ----------------
def centerline_curvature(Mz, voxel_nm = 8.0, pixel_nm=8.0, sigma_nm=24.0, smooth_nm=12.0, ds_nm=2.0,
                         min_length_nm=50.0, edge_trim_nm=10.0, positive_side='high',
                         z_slice=slice(None), plot=True, kappa_clim=0.02, verbose=True):
    """
    Signed in-plane curvature of the domain-wall centerline (the Mz = 0 contour of the
    z-averaged Mz), plus a rasterized kappa map on the pixel grid of Mz.

    Mz            : (nz, ny, nx) array
    pixel_nm      : in-plane pixel size of Mz
    sigma_nm      : Gaussian smoothing of the z-averaged Mz before finding the contour
                    (in nm, so the same value means the same thing at any pixel size;
                    sigma_nm=24 at 8 nm pixels equals your old sigma_pix=3)
    smooth_nm     : smoothing length along the contour before taking derivatives
    ds_nm         : spacing of the resampled contour points (independent of pixel size)
    min_length_nm : discard shorter contours
    edge_trim_nm  : open contours: set kappa to NaN this close to each end
    positive_side : 'high' -> kappa > 0 means the center of curvature is in the Mz > 0 domain
    z_slice       : layers to average, e.g. slice(z_lo, z_hi + 1)
    max_dist_nm   : rasterized pixels farther than this from the contour stay NaN

    Returns dict: contours, kappa_map, dist_map, Mz_2d_s, pixel_nm, nz, params,
                  all_x, all_y, all_k
    """
    Mz_2d = Mz[z_slice].mean(axis=0)
    Mz_2d_s = gaussian_filter(Mz_2d, sigma_nm / pixel_nm)
    ny, nx = Mz_2d_s.shape

    max_dist_nm = 10 * voxel_nm
    # ---- contour -> evenly spaced, smoothed points (nm) ----
    contours = []
    for c in measure.find_contours(Mz_2d_s, level=0.0):
        y_raw, x_raw = c[:, 0] * pixel_nm, c[:, 1] * pixel_nm
        closed = np.allclose(c[0], c[-1])
        s_raw = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_raw), np.diff(y_raw)))])
        L = s_raw[-1]
        if L < min_length_nm:
            continue
        n = int(round(L / ds_nm))
        s_new = np.linspace(0, L, n, endpoint=False) if closed else np.linspace(0, L, n + 1)
        x_r, y_r = np.interp(s_new, s_raw, x_raw), np.interp(s_new, s_raw, y_raw)
        ds = L / n
        mode = 'wrap' if closed else 'nearest'
        x_s = gaussian_filter1d(x_r, smooth_nm / ds, mode=mode)
        y_s = gaussian_filter1d(y_r, smooth_nm / ds, mode=mode)
        contours.append({'x': x_s, 'y': y_s, 'x_raw': x_r, 'y_raw': y_r,
                         'ds': ds, 'closed': closed, 'length_nm': L})
    if not contours:
        raise ValueError("No zero contour found. Check Mz, z_slice, and min_length_nm.")

    # ---- orient (domain on the left), then curvature ----
    want_positive = (positive_side == 'high')
    for i, c in enumerate(contours):
        x, y, ds, closed = c['x'], c['y'], c['ds'], c['closed']
        tx, ty = _d_ds(x, ds, closed), _d_ds(y, ds, closed)
        nrm = np.hypot(tx, ty)
        tx, ty = tx / nrm, ty / nrm
        off = 2 * pixel_nm
        xl, yl = x - ty * off, y + tx * off                      # point just left of the wall
        left_Mz = map_coordinates(Mz_2d_s, [yl / pixel_nm, xl / pixel_nm],
                                  order=1, mode='nearest')
        if (np.mean(left_Mz) > 0) != want_positive:
            x, y = x[::-1].copy(), y[::-1].copy()
            left_Mz = -left_Mz
        frac_ok = np.mean((left_Mz > 0) == want_positive)

        k = _signed_curvature(x, y, ds, closed)
        if not closed:
            n_trim = int(round(edge_trim_nm / ds))
            if n_trim > 0:
                k[:n_trim] = np.nan
                k[-n_trim:] = np.nan
        c['x'], c['y'], c['kappa'] = x, y, k

        if verbose:
            kv = k[np.isfinite(k)]
            rng_txt = f"[{kv.min():.4f}, {kv.max():.4f}]" if len(kv) else "n/a"
            print(f"contour {i}: {len(x)} pts, {c['length_nm']:.0f} nm, "
                  f"{'closed' if closed else 'open'}, {len(kv)} valid kappa, "
                  f"direction consistent {frac_ok:.0%}, kappa range {rng_txt} 1/nm")

    all_x = np.concatenate([c['x'] for c in contours])
    all_y = np.concatenate([c['y'] for c in contours])
    all_k = np.concatenate([c['kappa'] for c in contours])
    valid = np.isfinite(all_k)
    if not valid.any():
        raise ValueError("All kappa values are NaN (contours too short after edge trimming?).")
    if verbose:
        print("\nkappa summary (1/nm):")
        print(pd.Series(all_k).describe().to_string())

    # ---- rasterize kappa onto the pixel grid ----
    pts = np.column_stack([all_x[valid], all_y[valid]])
    yy, xx = np.mgrid[0:ny, 0:nx]
    grid_xy = np.column_stack([xx.ravel() * pixel_nm, yy.ravel() * pixel_nm])
    dist, j = cKDTree(pts).query(grid_xy)
    kappa_map = all_k[valid][j].reshape(ny, nx)
    dist_map = dist.reshape(ny, nx)
    kappa_map[dist_map > max_dist_nm] = np.nan

    params = dict(pixel_nm=pixel_nm, sigma_nm=sigma_nm, smooth_nm=smooth_nm, ds_nm=ds_nm,
                  min_length_nm=min_length_nm, edge_trim_nm=edge_trim_nm,
                  positive_side=positive_side, max_dist_nm=max_dist_nm,
                  z_slice=str(z_slice))
    result = dict(contours=contours, kappa_map=kappa_map, dist_map=dist_map,
                  Mz_2d_s=Mz_2d_s, pixel_nm=pixel_nm, nz=Mz.shape[0], params=params,
                  all_x=all_x, all_y=all_y, all_k=all_k)
    if plot:
        plot_centerline_kappa(result, kappa_clim)

    nz = volume['Mz'].shape[0]

    kappa_vol = np.repeat(np.nan_to_num(kappa_map)[None], nz, axis=0)             # NaN -> 0
    valid_vol = np.repeat(np.isfinite(kappa_map)[None].astype(float), nz, axis=0)
    dist_vol  = np.repeat(dist_map[None], nz, axis=0)

    num = sample_volume_at_points(kappa_vol, verts_idx)
    den = sample_volume_at_points(valid_vol, verts_idx)

    if 'kappa' in wall_df.columns:
        wall_df['kappa_old'] = wall_df['kappa']

    wall_df['kappa'] = np.where(den > 0.5, num / np.maximum(den, 1e-9), np.nan)
    wall_df['kappa_dist_nm'] = sample_volume_at_points(dist_vol, verts_idx)

    print(f"kappa is NaN for {wall_df['kappa'].isna().mean():.1%} of vertices")

    # top-down check
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(wall_df['x'], wall_df['y'], c=wall_df['kappa'], cmap='coolwarm',
                    s=1, vmin=-kappa_clim, vmax=kappa_clim)
    plt.colorbar(sc, label='κ (1/nm)')
    ax.set_aspect('equal')
    ax.set_title('Wall vertices colored by sampled κ (top-down)')
    plt.show()



    return result, wall_df


def plot_centerline_kappa(result, kappa_clim=0.02, title=None):
    p = result['pixel_nm']
    ny, nx = result['Mz_2d_s'].shape
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(result['Mz_2d_s'], cmap='viridis', origin='lower', alpha=0.6,
              extent=[-0.5 * p, (nx - 0.5) * p, -0.5 * p, (ny - 0.5) * p])
    sc = ax.scatter(result['all_x'], result['all_y'], c=result['all_k'], cmap='coolwarm',
                    s=6, vmin=-kappa_clim, vmax=kappa_clim)
    plt.colorbar(sc, label='κ (1/nm)')
    ax.set_xlabel('x (nm)')
    ax.set_ylabel('y (nm)')
    ax.set_title(f'Center line curvature\n{title}' if title else 'Center line curvature')
    ax.set_aspect('equal')
    plt.show()



In [ ]:

res = centerline_curvature(volume['Mz'], voxel_nm = voxel_nm, pixel_nm=voxel_nm, sigma_nm=24.0, smooth_nm=12.0, plot = False)



In [ ]:
sargs = dict(
    position_x=0.4,  # Move towards the right edge
    position_y=0.85,   # Move near the bottom
    height=0.05,       # Make it take up 50% of window height
    width=0.5,        # Width of the bar
    vertical=False,    # Orientation
    title = f'{sample_name} \n Kappa'
)

psi_plot_dict = dict(
    view= wall_df['kappa'].values, 
    bounds = [-0.02, 0.02],
    color_map = 'coolwarm', 
    zoom = 1
)

plot_map(volume, verts_xyz, faces, psi_plot_dict, sargs, show_grid = True, field = 'Mz', axis = 'z')

## Plot Bloch as a function of curvature

In [ ]:
def plot_z0_vs_curvature(z0_df, geom, n_bins=10, kappa_range=None, kappa_tol=0.002,
                         min_count=10, cluster_nm=40.0, normalize=False,
                         n_boot=500, seed=0, color='green', ylim=None, title=None,
                         bin_nm=8.0):
    """
    Left : mean z0 (nm from film center) vs. kappa, one value per column, standard errors.
    Right: distribution of z0 for kappa < -tol, |kappa| <= tol, kappa > tol (dashed = class mean).
    Returns (summary, class_summary, d), with the same columns as plot_bloch_v_curvature.
    """
    rng = np.random.default_rng(seed)
    d = z0_df.dropna(subset=['kappa']).copy()
    z = d['z0_rel_nm'].to_numpy()
    k = d['kappa'].to_numpy()
    w = np.ones(len(d))
    d['bx'] = np.floor(d['x'] / cluster_nm).astype(int)
    d['by'] = np.floor(d['y'] / cluster_nm).astype(int)
    cl = d.groupby(['bx', 'by']).ngroup().to_numpy()
    print(f"{len(d)} columns with valid kappa, {cl.max() + 1} bootstrap blocks of {cluster_nm:g} nm")

    if kappa_range is None:
        lim = np.percentile(np.abs(k), 98)
        kappa_range = (-lim, lim)
    print(f"kappa bins span {kappa_range[0]:.4f} to {kappa_range[1]:.4f} 1/nm")

    edges = np.linspace(*kappa_range, n_bins + 1)
    centers = 0.3 * (edges[:-1] + edges[1:])
    vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy()
    vm = ~np.isnan(vb)
    # mean, err, n_pts, n_cl, _ = _binned_cluster_stats(
    #     vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins, n_boot, rng)

    mean, err, n_pts, n_cl, ex = _binned_raw_stats(vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins, k=k[vm])

    keep = n_pts >= min_count
    summary = pd.DataFrame({'kappa_center': centers, 'z_nm_mean': mean, 'z_nm_err': err,
                            'n_points': n_pts, 'n_clusters': n_cl, 'shown': keep})

    cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
    names = [f'κ < -{kappa_tol:g}', f'|κ| ≤ {kappa_tol:g}', f'κ > {kappa_tol:g}']
    colors = ['tab:blue', 'tab:gray', 'tab:red']
    # c_mean, c_err, c_n, c_ncl, c_boots = _binned_cluster_stats(cls, z, w, cl, 3, n_boot, rng)
    c_mean, c_err, c_n, c_ncl, c_ex = _binned_raw_stats(cls, z, w, cl, 3, k = None)


    class_summary = pd.DataFrame({'class': names, 'z_nm_mean': c_mean, 'z_nm_err': c_err,
                                  'n_points': c_n, 'n_clusters': c_ncl})

    fig, (ax, axh) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax.errorbar(centers[keep], mean[keep], yerr=err[keep], fmt='o-', capsize=3, color=color)
    ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
    ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
    
    for j, c in enumerate(colors):
        ax.axhline(c_mean[j], color=c, ls='--', lw=1.5)



    # ax.axvspan(-kappa_tol, kappa_tol, color='gray', alpha=0.12, label=f'|κ| ≤ {kappa_tol:g}')
    ax.set_xlabel('dw curvature (1/nm)')
    ax.set_ylabel('M_normal = 0 (Bloch) position relative to film center (nm)')
    ax.set_title(f'Mean Bloch position (z) vs. curvature \n {title}')
    ax.legend(fontsize=8)
    if ylim is not None:
        ax.set_ylim(*ylim)

    # x_vals, vox = geom['x_vals'], geom['voxel_nm']
    # zedges = np.concatenate([x_vals - vox / 2, [x_vals[-1] + vox / 2]])
    # wb = 0.27 * vox
    # for j, c in enumerate(colors):
    #     m = cls == j
    #     counts, _ = np.histogram(z[m], bins=zedges)
    #     vals = counts / counts.sum() if normalize and counts.sum() > 0 else counts
    #     axh.bar(x_vals + (j - 1) * wb, vals, width=wb, color=c, label=f'{names[j]} (n={m.sum()})')
    #     axh.axvline(c_mean[j], color=c, ls='--', lw=1.5)
    # axh.axvline(0, color='k', ls=':', lw=0.8)
    # axh.set_xlabel('z relative to film center (nm)')
    # axh.set_ylabel('fraction of columns' if normalize else '# columns')
    # axh.set_xticks(x_vals)
    # axh.set_xticklabels([f'{v:g}' for v in x_vals], rotation=45)

    x_vals, vox = geom['x_vals'], geom['voxel_nm']
    z_lo = x_vals.min() - vox / 2
    z_hi = x_vals.max() + vox / 2
    z_edges = np.arange(z_lo, z_hi + bin_nm, bin_nm)
    z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
    wb = 0.3 * bin_nm                      # width of each of the 3 grouped bars

    for j, c in enumerate(colors):
        m = cls == j
        counts, _ = np.histogram(z[m], bins=z_edges)
        vals = counts / counts.sum() if normalize and counts.sum() > 0 else counts
        axh.bar(z_centers + (j - 1) * wb, vals, width=wb, color=c,
                label=f'{names[j]} (n={m.sum()})')
        axh.axvline(c_mean[j], color=c, ls='--', lw=1.5)
    axh.axvline(0, color='k', ls=':', lw=0.8)
    axh.set_xlabel('z relative to film center (nm)')
    axh.set_ylabel('fraction of layers' if normalize else 'counts')
    axh.set_xticks(z_centers)
    axh.set_xticklabels([f'{v:g}' for v in z_centers], rotation=45)

    axh.set_title('Bloch position distribution by curvature')
    axh.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\nClass summary (one value per column, block-bootstrap error; z in nm from center):")
    print(class_summary.to_string(index=False))
    # diff = c_boots[:, 2] - c_boots[:, 0]
    # print(f"\nmean z0(κ>tol) - mean z0(κ<-tol) = {np.nanmean(diff):.2f} ± {np.nanstd(diff):.2f} nm")
    return summary, class_summary, d

def Investigate_kappa_distribution (grouped, z0_df,  color = 'green', voxel_nm = 2.0, bin_nm = 8.0, scatter_key = 'kappa' ):

    x_vals = grouped['z_rel_nm'].to_numpy()
    z = z0_df['z0_rel_nm'].to_numpy()
    # zedges = np.concatenate([x_vals - voxel_nm / 2, [x_vals[-1] + voxel_nm / 2]])

    lo = x_vals.min() - voxel_nm / 2
    hi = x_vals.max() + voxel_nm / 2
    zedges = np.arange(lo, hi + bin_nm, bin_nm)       # + bin_nm so the last edge covers hi
    centers = 0.5 * (zedges[:-1] + zedges[1:])

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.scatter(z0_df[scatter_key], z, c=color,  s=15, alpha=0.6)   # <-- change 'x' to your column
    ax.axhline(0, color='gray', ls=':', label = 'film center')
    # ax[1,0].axhline(stats['mean'], color='red', ls='--', label=f"mean = {stats['mean']:.2f}")
    ax.set_xlabel(scatter_key, fontsize=14)
    ax.set_ylabel('z relative to film center (nm)', fontsize=14)
    ax.set_title(f'z vs. {scatter_key}')
    # ax[1,0].set_ylim(np.min(xs), np.max(xs) )

    ax.legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
z0_df, misc_df, crossings = find_M0_crossings_mesh(
    wall_df, faces, geom, angle_col='M_n_n', voxel_nm=voxel_nm)

print("z0_df kappa valid:", z0_df['kappa'].notna().sum(), "of", len(z0_df))

z0_summary, z0_class_summary, z0_pts = plot_z0_vs_curvature(
    z0_df, geom, n_bins = 7,  kappa_tol=0.005, normalize=False, color=sample_color, title=f'sample {sample_name}', bin_nm=voxel_nm)

#Investigate_kappa_distribution (grouped, z0_df,  color = sample_color, voxel_nm = 2.0, bin_nm = 8.0, scatter_key = 'kappa' )

In [ ]:
z0_summary, z0_class_summary, z0_pts = plot_z0_vs_curvature(
    z0_df, geom, n_bins = 5,  kappa_tol=0.005, normalize=False, color=sample_color, title=f'sample {sample_name}', bin_nm=voxel_nm)

In [ ]:
___, ___, ___, ___, ___ = plot_Bloch_angle_vs_z_m0(
    wall_df, faces, geom=geom, voxel_nm=voxel_nm, bin_nm=8.0, drop_degenerate=False, color=sample_color, title=f'{sample_name}')

# save data_frame

In [ ]:
stop

In [ ]:
def make_json_safe(obj):
    """
    Recursively convert numpy types within a nested dict/list/tuple
    structure into native Python types, so it can be safely passed to
    json.dump. Handles numpy ints, floats, bools, and arrays.
    """
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    else:
        return obj

In [ ]:
def save_sample_data(save_dir, sample_name, wall_df, z0_df,
                      spacing=(2.0, 2.0, 2.0), z_offset=0, date = None,
                      extra_metadata=None):
    """
    Save the cropped volume and wall-analysis DataFrame for one sample,
    in a format suitable for later cross-sample/material comparison.

    Parameters
    ----------
    save_dir : str
        Directory to save into (created if it doesn't exist).
    sample_name : str
        Identifier for this sample/material, used in filenames and
        stamped into the saved data for later grouping.
    wall_df : pd.DataFrame
        The per-vertex wall analysis table (normals, psi, chirality, etc.)
    spacing : tuple
        (dz, dy, dx) physical voxel size, in nm.
    z_offset : int
        Offset applied to bring local z-indices back into the original
        volume's index frame (as discussed earlier).
    extra_metadata : dict or None
        Any additional key-value info you want stored alongside (e.g.
        material composition, deposition conditions, notes).
    """
    os.makedirs(save_dir, exist_ok=True)


    # --- Save wall_df, tagged with sample identity ---
    df_out = wall_df.copy()
    df_out['sample_name'] = sample_name
    df_path = os.path.join(save_dir, f'{sample_name}_wall_df_{date}.csv')
    df_out.to_csv(df_path, index=False)

    # --- Save human-readable metadata alongside, for provenance ---
    metadata = {
        'sample_name': sample_name,
        'spacing_nm': list(spacing),
        'z_offset': z_offset,
        'n_wall_vertices': len(wall_df),
        'volume_shape': list(volume['shape']),
    }


    # --- Save z0_df, tagged with sample identity ---
    zdf_out = z0_df.copy()
    zdf_out['sample_name'] = sample_name
    zdf_path = os.path.join(save_dir, f'{sample_name}_z0_df_{date}.csv')
    zdf_out.to_csv(zdf_path, index=False)



    if extra_metadata:
        metadata.update(extra_metadata)
    meta_path = os.path.join(save_dir, f'{sample_name}_wall_analysis_metadata_{date}.json')
    with open(meta_path, 'w') as f:
        json.dump(make_json_safe(metadata), f, indent=2)

    print(f"Saved: {df_path}\n       {meta_path} \n {zdf_path}")

    return df_path, meta_path, zdf_path

In [ ]:
save_dir = data_dir
date_save = date

print(date_save)

In [ ]:

save_sample_data(save_dir, sample_name, wall_df, z0_df, spacing=(dz, dy, dx), z_offset=0, date = date_save, extra_metadata=None)

## Save for paraview

In [ ]:
def export_wall_mesh_to_paraview(verts_xyz, faces, wall_df, save_path,
                                  scalar_cols=None, vector_col_groups=None):
    """
    Export the domain wall isosurface mesh, with all computed scalar and
    vector fields attached as point data, to a .vtp file readable
    directly in ParaView.

    Parameters
    ----------
    verts_xyz : (N, 3) array
        Mesh vertex coordinates, as produced by marching_cubes.
    faces : (M, 3) array
        Triangle connectivity, as produced by marching_cubes.
    wall_df : pd.DataFrame
        Per-vertex table (must have the same row order/length as
        verts_xyz) containing scalar and vector components to export.
    save_path : str
        Output file path, should end in '.vtp'.
    scalar_cols : list of str or None
        Column names in wall_df to attach as scalar point-data fields
        (e.g. ['psi_unsigned', 'psi_chiral', 'curl_magnitude',
        'dmi_interfacial']). If None, a sensible default set is used
        if present in wall_df.
    vector_col_groups : dict of {name: (x_col, y_col, z_col)} or None
        Named vector fields to attach, built from three wall_df columns
        each, e.g. {'Normal': ('nx','ny','nz'), 'Tangent': ('tx','ty','tz'),
        'M': ('Mx','My','Mz')}. If None, a sensible default set is used
        if present in wall_df.

    Returns
    -------
    mesh : pv.PolyData
        The mesh object that was saved, in case you want to inspect or
        further modify it in the same session.
    """
    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
    mesh = pv.PolyData(verts_xyz, faces_pv)

    if scalar_cols is None:
        candidates = ['psi_unsigned', 'psi_chiral', 'curl_magnitude',
                      'dmi_interfacial', 'divergence', 'category']
        scalar_cols = [c for c in candidates if c in wall_df.columns]

    if vector_col_groups is None:
        candidates = {
            'Normal': ('nx', 'ny', 'nz'),
            'Tangent': ('tx', 'ty', 'tz'),
            'M': ('Mx', 'My', 'Mz'),
        }
        vector_col_groups = {
            name: cols for name, cols in candidates.items()
            if all(c in wall_df.columns for c in cols)
        }

    for col in scalar_cols:
        values = wall_df[col].values
        if values.dtype == object:  # e.g. 'category' being string labels
            # ParaView can't handle string arrays as colorable scalars directly;
            # encode as integer codes instead, and keep a lookup printed for reference.
            codes, uniques = pd.factorize(values)
            mesh.point_data[f'{col}_code'] = codes.astype(np.float32)
            print(f"'{col}' is categorical; saved as '{col}_code' with mapping: "
                  f"{dict(enumerate(uniques))}")
        else:
            mesh.point_data[col] = values.astype(np.float32)

    for vec_name, (cx, cy, cz) in vector_col_groups.items():
        vec_array = wall_df[[cx, cy, cz]].values.astype(np.float32)
        mesh.point_data[vec_name] = vec_array

    mesh.save(save_path)
    print(f"Saved mesh with {mesh.n_points} points, {mesh.n_cells} cells to {save_path}")
    print(f"  Scalar fields: {[c for c in scalar_cols]}")
    print(f"  Vector fields: {list(vector_col_groups.keys())}")

    return mesh

In [ ]:

save_path = os.path.join(data_dir, f'{sample_name}_wall_mesh_{date_save}.vtp')

mesh = export_wall_mesh_to_paraview(
    verts_xyz, faces, wall_df,
    save_path=save_path,
    scalar_cols=['M_normal', 'psi_unsigned', 'psi_chiral', 'Mx', 'My'],
    vector_col_groups={'Normal': ('nx', 'ny', 'nz'),
                        'Tangent': ('tx', 'ty', 'tz'),
                        'M': ('Mx', 'My', 'Mz')}
)

In [ ]:
stop

In [ ]:
# save_path = os.path.join(data_dir, f'{sample_name}_div_wall_mesh_{date_save}.vtp')

# div_mesh = export_wall_mesh_to_paraview(
#     verts_xyz, faces, wall_df,
#     save_path=save_path,
#     scalar_cols=['div_M','source', 'sink'],
#     vector_col_groups={'Normal': ('nx', 'ny', 'nz'),
#                         'Tangent': ('tx', 'ty', 'tz'),
#                         'M': ('Mx', 'My', 'Mz')}
# )

save cropped volume

In [ ]:
volume.keys()

In [ ]:
date_save

In [ ]:

# Swap array axes: (nz, ny, nx) -> (nx, ny, nz)
Mx = volume["Mx"].transpose(2, 1, 0)
My = volume["My"].transpose(2, 1, 0)
Mz = volume["Mz"].transpose(2, 1, 0)
magnitude = volume["magnitude"].transpose(2, 1, 0)

nx, ny, nz = Mx.shape

dx = dy = dz = 1.0   # voxel size, if known
x = np.arange(nx) * dx
y = np.arange(ny) * dy
z = np.arange(nz) * dz
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

grid = pv.StructuredGrid(X, Y, Z)

grid.point_data["M"] = np.stack([Mx.flatten(order="F"),
                                 My.flatten(order="F"),
                                 Mz.flatten(order="F")], axis=1)
grid.point_data["Mx"] = Mx.flatten(order="F")
grid.point_data["My"] = My.flatten(order="F")
grid.point_data["Mz"] = Mz.flatten(order="F")
grid.point_data["magnitude"] = magnitude.flatten(order="F")

grid.save(f"{sample_name}_{date_save}_cropped.vtk")




In [ ]:
stop